# Thresholds and Curves

**Notebook 4 of 5.** Logistic regression does not predict fraud or honest. It predicts a
**probability** of fraud, and `predict` turns that into a class with a fixed cut-off of
0.5. Every score in notebook 03 was computed at that one threshold, which nobody chose.

This notebook moves the threshold, watches precision and recall trade places, and meets
the two curves that summarize a model across all thresholds at once.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    precision_score, recall_score, fbeta_score, confusion_matrix,
    precision_recall_curve, PrecisionRecallDisplay, RocCurveDisplay,
    average_precision_score, roc_auc_score,
)

The same steps as in notebook 01, in one cell: load, split, scale, and train a logistic
regression.

In [ ]:
df = pd.read_csv("data/transactions.csv")
X = df.drop(columns="fraud")
y = df["fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000).fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

# the probability of fraud for every test payment
proba_test = model.predict_proba(X_test_scaled)[:, 1]

## 1. Moving the threshold

A lower threshold raises more alarms: more frauds caught, and more honest payments
flagged. The table shows the model at five thresholds.

In [ ]:
rows = []
for threshold in [0.9, 0.5, 0.3, 0.1, 0.05]:
    pred = (proba_test >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "alarms": pred.sum(),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "F2": fbeta_score(y_test, pred, beta=2),
    })

pd.DataFrame(rows).round(3)

Read it top to bottom. As the threshold falls, recall rises and precision falls: **the
threshold is the dial between the two mistakes.** The model stays the same; only the
decision made from its probabilities changes.

Which threshold is right is the question from notebook 02 again: which mistake costs more.

## 2. Choosing a threshold, without the test set

Say the bank wants to catch **at least 80 percent of frauds**. The tempting move is to
read the answer off the table above. But that table was computed on the test set, and
choosing anything with the test set is test leakage: the test score would no longer be a
fair measurement.

So the threshold is chosen on the training set. `precision_recall_curve` returns the
precision and recall at every possible threshold:

In [ ]:
proba_train = model.predict_proba(X_train_scaled)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_train, proba_train)

# the last precision and recall belong to no threshold, so they are dropped
candidates = pd.DataFrame(
    {"threshold": thresholds, "precision": precision[:-1], "recall": recall[:-1]}
)
# a few of them, from low thresholds to high
candidates.iloc[::6000].round(3)

> **Exercise:**
>
> From `candidates`, pick the **highest threshold whose training recall is at least
> 0.8**. Then apply that threshold to `proba_test`, once, and report the test recall and
> precision. Did the target hold up on new data?

<details><summary>
Click here for a hint...
</summary>

Filter with `candidates[candidates["recall"] >= 0.8]`, then take the largest value of its
`threshold` column with `.max()`.
</details>

In [ ]:
# your code: choose the threshold on the training set, then test it once

<details><summary>
Click here to compare your answer...
</summary>

The threshold comes out at about **0.39**, and on the test set recall is **0.797**: just
short of the 0.8 target. That is what honest evaluation looks like: a target met on the
training data usually slips a little on new data. A model that seemed to hit it exactly on
the test set would deserve suspicion.

With 123 test frauds, one more caught would have reached the target. Choosing thresholds
on a separate **validation set** is the next refinement, later in the program.
</details>

## 3. All thresholds at once: two curves

Instead of a table, plot precision against recall for every threshold. That is the
**precision-recall curve**:

In [ ]:
PrecisionRecallDisplay.from_predictions(y_test, proba_test, name="logistic regression")
plt.title("Precision and recall at every threshold");

A perfect model would sit in the top right corner. The area under this curve, the
**average precision** or PR AUC, summarizes the whole curve in one number between 0 and 1.

The other common curve is the **ROC curve**. It plots recall, here called the true
positive rate, against the **false positive rate**: the share of honest payments that
were flagged, FP / (FP + TN).

In [ ]:
RocCurveDisplay.from_predictions(y_test, proba_test, name="logistic regression")
plt.plot([0, 1], [0, 1], "k:", label="random guessing")
plt.legend()
plt.title("The ROC curve");

In [ ]:
print(f"ROC AUC:           {roc_auc_score(y_test, proba_test):.3f}   random guessing: 0.5")
print(f"average precision: {average_precision_score(y_test, proba_test):.3f}   random guessing: {y_test.mean():.3f}")

**The ROC curve looks almost perfect, and on imbalanced data that is misleading.** Its
false positive rate divides by all 12,377 honest payments, so even many false alarms
barely move it. The exercise makes that concrete.

> **Exercise:**
>
> At a threshold of **0.05**, count the false positives and compute the false positive
> rate. Compare the number of false alarms with the number of frauds caught.

In [ ]:
# your code: false alarms and the false positive rate at threshold 0.05

<details><summary>
Click here to compare your answer...
</summary>

**38 false alarms against 106 frauds caught**: more than one alarm in four is a false one,
which a fraud team would certainly notice. Yet the false positive rate is only **0.003**,
because 38 is a tiny share of 12,377 honest payments. The ROC curve sees 0.003 and stays
in the top left corner; precision sees 38 out of 144 alarms.

That is why, on imbalanced data, **the precision-recall curve and average precision are
the more honest summary**. ROC AUC is still widely reported, and fine for comparing
models on balanced data.
</details>

## Summary

- A classifier outputs probabilities; **the threshold turns them into decisions**, and
  moving it trades precision against recall.
- **Choose the threshold on the training data**, never on the test set.
- The **precision-recall curve** and its area, **average precision**, summarize a model at
  every threshold. On imbalanced data they are more honest than the **ROC curve** and ROC
  AUC, which the many true negatives flatter.

Next, in `05_handling_imbalance.ipynb`: instead of only measuring the imbalance, doing
something about it.